# Accessing the Complete SymbTr v3.0 Dataset

This notebook presents a reproducible workflow for accessing the complete **SymbTr v3.0 dataset** from its official Zenodo repository.

The SymbTr collection is distributed through Zenodo as a set of published resources, including symbolic music representations and accompanying metadata. Before performing any computational analysis, it is essential to obtain the original dataset from its official source and verify that all resources have been downloaded correctly.

This notebook retrieves the Zenodo record metadata, inspects the published resources, downloads all available dataset archives, extracts their contents, and verifies the resulting local directory structure. Establishing this acquisition workflow ensures that every subsequent notebook operates on the same verified version of the dataset, thereby improving reproducibility and transparency.

The following notebooks will investigate the downloaded resources individually, beginning with the TXT collection and subsequently exploring the XML representation of Turkish makam music.

## Importing the Required Libraries

This notebook uses a small collection of Python libraries to download, organize, and inspect the SymbTr v3.0 dataset.

The `Path` class from the `pathlib` module provides a platform-independent way to manage project directories and file paths. The `requests` library is used to communicate with the Zenodo REST API and retrieve dataset metadata, while the `zipfile` module enables extraction of compressed dataset archives. Finally, `pandas` is employed for displaying and manipulating metadata in a structured tabular format throughout the notebook.

**Output.** This cell imports the Python libraries required for data acquisition, archive management, and metadata exploration.

In [13]:
from pathlib import Path
import zipfile

import pandas as pd
import requests

## Defining the Zenodo Record and Project Data Directories

The SymbTr v3.0 dataset is publicly available through the Zenodo research data repository. Each Zenodo publication is assigned a unique record identifier, which can be accessed programmatically through the Zenodo REST API.

In this section, we define the Zenodo record identifier together with the corresponding API endpoint. The notebook also initializes the standard project data structure by creating dedicated directories for **raw**, **interim**, and **processed** data. Organizing datasets in this manner promotes reproducibility, improves project organization, and ensures that all subsequent notebooks access the required resources through a consistent directory structure.

If the required project directories do not already exist, they are created automatically before any dataset is downloaded or processed.

**Output.** This cell initializes the standard project data directories, defines the Zenodo record identifier and API endpoint, and prepares the project environment for the subsequent data acquisition and analysis steps.

In [14]:
# Zenodo record identifier
record_id = "15470412"

# Zenodo REST API endpoint
api_url = f"https://zenodo.org/api/records/{record_id}"

# Resolve the project root directory
project_root = Path.cwd().resolve().parents[1]

# Define standard data directories
data_directory = project_root / "data"
raw_data_directory = data_directory / "raw"
interim_data_directory = data_directory / "interim"
processed_data_directory = data_directory / "processed"

# Create the directories if they do not already exist
for directory in (
    raw_data_directory,
    interim_data_directory,
    processed_data_directory,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Project data directories have been initialized successfully.")

Project data directories have been initialized successfully.


## Retrieving the Dataset Metadata

Before downloading the dataset, the notebook retrieves the associated metadata from the Zenodo repository using its REST API. The metadata contain essential information about the dataset, including its title, publication details, licensing information, and downloadable resources.

The metadata are returned in JSON format and converted into a Python dictionary, allowing subsequent notebook cells to access the dataset programmatically. If the request is unsuccessful, an exception is raised to ensure that the workflow stops before attempting to download unavailable resources.

**Output.** This cell retrieves the SymbTr v3.0 metadata from the Zenodo REST API, validates the server response, stores the metadata as a Python dictionary, and confirms that the retrieval process has completed successfully.

In [15]:
# Retrieve the dataset metadata from Zenodo
response = requests.get(api_url, timeout=30)

# Raise an exception if the request was unsuccessful
response.raise_for_status()

# Convert the JSON response into a Python dictionary
zenodo_record = response.json()

print("Dataset metadata successfully retrieved.")

Dataset metadata successfully retrieved.


## Exploring the Dataset Metadata Structure

Before extracting specific information from the Zenodo record, it is useful to examine the overall structure of the retrieved metadata. The Zenodo REST API returns the dataset description as a hierarchical JSON object containing bibliographic information, persistent identifiers, downloadable files, licensing details, and additional repository metadata.

The code below lists the available top-level metadata fields. Inspecting these fields provides an overview of the dataset structure and helps identify the information that will be used throughout the subsequent metadata extraction process.

**Output.** This cell displays the top-level metadata fields available in the SymbTr v3.0 Zenodo record.

In [16]:
# Display the available top-level metadata fields
metadata_fields = list(zenodo_record.keys())

metadata_fields

['created',
 'modified',
 'id',
 'conceptrecid',
 'doi',
 'conceptdoi',
 'doi_url',
 'metadata',
 'title',
 'links',
 'updated',
 'recid',
 'revision',
 'files',
 'swh',
 'owners',
 'status',
 'stats',
 'state',
 'submitted']

## Inspecting the Dataset Metadata

Before downloading the dataset resources, it is useful to examine the descriptive metadata provided by Zenodo. These metadata summarize the publication and confirm that the correct version of the dataset has been selected.

In this section, the notebook extracts key bibliographic information from the retrieved metadata, including the dataset title, Digital Object Identifier (DOI), version, publication date, license, and publisher. Reviewing these metadata helps verify the authenticity, provenance, and reproducibility of the dataset that will be used throughout the remaining notebooks.

For improved readability, the selected metadata are presented as a structured table using a Pandas DataFrame.

**Output.** This cell displays the principal bibliographic metadata describing the SymbTr v3.0 dataset in a structured table.

In [17]:
# Extract selected bibliographic metadata
metadata = {
    "Title": zenodo_record["metadata"]["title"],
    "DOI": zenodo_record["doi"],
    "Version": zenodo_record["metadata"].get(
        "version",
        "Not specified",
    ),
    "Publication Date": zenodo_record["metadata"]["publication_date"],
    "License": zenodo_record["metadata"]["license"]["id"],
    "Publisher": zenodo_record.get(
        "publisher",
        "Zenodo",
    ),
}

# Create the metadata table
metadata_table = pd.DataFrame(
    metadata.items(),
    columns=["Field", "Value"],
)

metadata_table.index = range(
    1,
    len(metadata_table) + 1,
)

# Display the table
display(
    metadata_table.style
    .hide(axis="index")
    .set_properties(
        **{
            "text-align": "left",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th.col_heading",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("width", "100%"),
                ],
            },
        ]
    )
)

Field,Value
Title,Turkish maqam music symbolic data (SymbTr v3.0)
DOI,10.5281/zenodo.15470412
Version,Not specified
Publication Date,2016-10-13
License,cc-by-4.0
Publisher,Zenodo


## Inspecting the Dataset Creators

In addition to the bibliographic metadata, Zenodo records provide information about the individuals who created or contributed to the dataset. Identifying the dataset creators is important for proper attribution, citation, and understanding the provenance of the published resource.

In this section, the notebook extracts the creator information from the Zenodo metadata and presents it in a structured table using a Pandas DataFrame. Depending on the metadata available in the Zenodo record, the table may include the creators' names, affiliations, ORCID identifiers, and other descriptive information.

**Output.** This cell displays the creators associated with the SymbTr v3.0 dataset in a structured table.

In [18]:
from pathlib import Path
import pandas as pd

# Extract the published resources from the Zenodo record
published_resources = pd.DataFrame(
    [
        {
            "Filename": resource["key"],
            "Size (MB)": resource["size"] / (1024 ** 2),
        }
        for resource in zenodo_record["files"]
    ]
)

# Summarize the published resources
resource_summary = pd.DataFrame(
    {
        "Metric": [
            "Number of published resources",
            "Total size (MB)",
            "Number of compressed archives",
        ],
        "Value": [
            len(published_resources),
            published_resources["Size (MB)"].sum(),
            (
                published_resources["Filename"]
                .astype(str)
                .str.lower()
                .str.endswith(".zip")
                .sum()
            ),
        ],
    }
)

display(
    resource_summary.style
    .hide(axis="index")
    .format(
        {
            "Value": lambda value: (
                f"{value:.2f}".rstrip("0").rstrip(".")
                if isinstance(value, float)
                else str(value)
            )
        }
    )
    .set_properties(
        **{
            "text-align": "left",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                    ("vertical-align", "top"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("width", "100%"),
                ],
            },
        ]
    )
)

# Determine the distribution of published file formats
file_format_distribution = (
    published_resources["Filename"]
    .astype(str)
    .apply(lambda filename: Path(filename).suffix.lower())
    .replace("", "No extension")
    .value_counts()
    .rename_axis("File Format")
    .reset_index(name="Number of Files")
)

display(
    file_format_distribution.style
    .hide(axis="index")
    .set_properties(
        **{
            "text-align": "left",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                    ("vertical-align", "top"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("width", "100%"),
                ],
            },
        ]
    )
)

Metric,Value
Number of published resources,6
Total size (MB),570.46
Number of compressed archives,5


File Format,Number of Files
.zip,5
.csv,1


**Interpretation.** The SymbTr v3.0 Zenodo record contains both compressed archive files and a CSV file. The ZIP archives preserve the original dataset resources in multiple symbolic formats, whereas the CSV file provides a structured tabular representation of the symbolic music data together with the associated metadata. This representation enables efficient statistical analysis, visualization, and machine learning workflows without requiring researchers to parse the original symbolic notation files, and was prepared to support computational studies on Turkish makam music {cite}`Bozkurt2014AAWM`. 

## Downloading the Published Dataset Resources

After inspecting the published resources, the dataset files can be downloaded directly from the Zenodo repository. Rather than downloading each resource manually, the notebook automatically retrieves every published file listed in the Zenodo metadata.

Each resource is downloaded only if it is not already available in the project's **raw data** directory. This avoids unnecessary network requests, reduces execution time when the notebook is executed multiple times, and supports reproducible data management.

To improve reliability and memory efficiency, the files are downloaded using streamed HTTP requests, allowing each file to be written incrementally to disk while preserving its original filename.

**Output.** This cell downloads all published dataset resources to the project's raw data directory and reports the status of each download.

In [20]:
from pathlib import Path
import pandas as pd
import requests

# Store download results
download_results = []

# Download counters
downloaded_files = 0
existing_files = 0
failed_files = 0

# Download every resource listed in the Zenodo metadata
for resource in zenodo_record["files"]:

    filename = resource["key"]

    download_url = (
        resource.get("links", {}).get("self")
        or resource.get("links", {}).get("download")
    )

    target_file = raw_data_directory / filename

    if target_file.exists():
        status = "Existing"
        existing_files += 1

    elif not download_url:
        status = "Failed: Download URL not available"
        failed_files += 1

    else:
        try:
            with requests.get(
                download_url,
                stream=True,
                timeout=120,
            ) as response:

                response.raise_for_status()

                with target_file.open("wb") as output_file:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            output_file.write(chunk)

            status = "Downloaded"
            downloaded_files += 1

        except requests.RequestException as error:
            status = f"Failed: {type(error).__name__}"
            failed_files += 1

            # Remove incomplete files
            if target_file.exists():
                target_file.unlink()

    download_results.append(
        {
            "Filename": filename,
            "Status": status,
        }
    )

# Create the download summary table
download_results_df = pd.DataFrame(
    download_results,
    columns=[
        "Filename",
        "Status",
    ],
)

display(
    download_results_df.style
    .hide(axis="index")
    .set_properties(
        **{
            "text-align": "left",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                    ("vertical-align", "top"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("width", "100%"),
                ],
            },
        ]
    )
)

print("\nDownload process completed.")
print("-" * 40)
print(f"Downloaded files : {downloaded_files}")
print(f"Existing files   : {existing_files}")
print(f"Failed files     : {failed_files}")
print(f"Total resources  : {len(download_results_df)}")
print("Files saved to   : data/raw")

Filename,Status
mid_v3.zip,Existing
Turkish maqam music pieces in time series format_3000 rows x 365 columns.csv,Existing
txt_v3.zip,Existing
mu2_v3.zip,Existing
xml_v3.zip,Existing
pdf_v3.zip,Existing



Download process completed.
----------------------------------------
Downloaded files : 0
Existing files   : 6
Failed files     : 0
Total resources  : 6
Files saved to   : data/raw


## Inspecting the Contents of the ZIP Archives

Before extracting the downloaded archives, it is useful to examine their contents. This inspection provides an overview of the files stored in each archive and confirms that the expected dataset resources are available.

For each ZIP archive, the notebook records the total number of files and summarizes the detected file types without extracting the archive. This overview helps identify the contents of the published resources before the extraction process begins.

**Output.** This cell summarizes the contents of each ZIP archive, including the number of files and the detected file types.

In [21]:
import zipfile

import pandas as pd

# Inspect the contents of each ZIP archive
archive_summary = []

for archive in sorted(raw_data_directory.iterdir()):

    if archive.is_file() and zipfile.is_zipfile(archive):

        with zipfile.ZipFile(archive, "r") as zip_file:

            members = zip_file.namelist()

            file_extensions = (
                pd.Series(members)
                .str.lower()
                .str.extract(
                    r"(\.[a-z0-9]+)$",
                    expand=False,
                )
                .fillna("No extension")
            )

            extension_summary = (
                file_extensions
                .value_counts()
                .sort_index()
            )

            archive_summary.append(
                {
                    "Archive": archive.name,
                    "Number of Files": len(members),
                    "Contained File Types": ", ".join(
                        [
                            f"{extension} ({count})"
                            for extension, count in extension_summary.items()
                        ]
                    ),
                }
            )

archive_summary_df = pd.DataFrame(archive_summary)

archive_summary_df.index = range(
    1,
    len(archive_summary_df) + 1,
)

display(
    archive_summary_df.style
    .set_properties(
        **{
            "text-align": "left",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                    ("vertical-align", "top"),
                ],
            },
            {
                "selector": "th.row_heading",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "normal"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("width", "100%"),
                ],
            },
        ]
    )
)

print("\nZIP archive inspection completed successfully.")
print("-" * 40)
print(f"ZIP archives inspected : {len(archive_summary_df)}")

,Archive,Number of Files,Contained File Types
1,mid_v3.zip,3001,".mid (3000), No extension (1)"
2,mu2_v3.zip,3001,".mu2 (3000), No extension (1)"
3,pdf_v3.zip,3001,".csv (1), .pdf (2999), No extension (1)"
4,txt_v3.zip,3001,".txt (3000), No extension (1)"
5,xml_v3.zip,3001,".xml (3000), No extension (1)"



ZIP archive inspection completed successfully.
----------------------------------------
ZIP archives inspected : 5


## Extracting the Dataset Archives

After downloading the dataset resources, the ZIP archives are extracted into the project's **interim data** directory.

Each archive is extracted into a dedicated subdirectory named after the archive. Before extraction, any existing directory with the same name is removed to ensure that the extracted contents always reflect the latest version of the downloaded archive. This approach prevents outdated files from remaining in the project and guarantees a clean and reproducible extraction process.

Maintaining the extracted files in the **interim** directory preserves the original organization of the published dataset while preparing the resources for subsequent preprocessing and analysis.

**Output.** This cell extracts all downloaded ZIP archives into the project's interim data directory and reports the extraction status of each archive.

In [22]:
from pathlib import Path
import shutil
import zipfile
import pandas as pd

# Locate all downloaded ZIP archives
zip_archives = sorted(
    raw_data_directory.glob("*.zip")
)

# Store extraction results
extraction_results = []

# Extract each archive
for archive in zip_archives:

    extraction_directory = (
        interim_data_directory
        / archive.stem
    )

    # Remove previous extraction if it exists
    if extraction_directory.exists():
        shutil.rmtree(extraction_directory)

    extraction_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        archive,
        "r",
    ) as zip_file:

        zip_file.extractall(
            extraction_directory
        )

    extraction_results.append(
        {
            "Archive": archive.name,
            "Status": "Extracted",
        }
    )

# Create the extraction summary table
extraction_summary = pd.DataFrame(
    extraction_results
)

display(
    extraction_summary.style
    .hide(axis="index")
    .set_properties(
        **{
            "text-align": "left",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                    ("vertical-align", "top"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("width", "100%"),
                ],
            },
        ]
    )
)

print("\nArchive extraction completed.")
print("-" * 40)
print(f"Archives processed : {len(extraction_summary)}")
print(f"Extracted          : {len(extraction_summary)}")
print("Files extracted to : data/interim")

Archive,Status
mid_v3.zip,Extracted
mu2_v3.zip,Extracted
pdf_v3.zip,Extracted
txt_v3.zip,Extracted
xml_v3.zip,Extracted



Archive extraction completed.
----------------------------------------
Archives processed : 5
Extracted          : 5
Files extracted to : data/interim


# Summarizing the Extracted File Formats

After extracting the dataset archives, it is useful to examine the distribution of the extracted file formats. This provides an overview of the resources available for subsequent analyses and verifies that the extraction process completed successfully.

The file extensions are collected from all files contained in the project's **interim data** directory and grouped according to their format. This summary helps identify the types of resources included in the dataset, such as symbolic music files, metadata, documentation, images, or other supplementary materials.

**Output.** This cell summarizes the extracted files by their file extensions and reports the number of files available for each format.

In [23]:
from pathlib import Path
import pandas as pd

# Collect all extracted files
all_files = [
    file
    for file in interim_data_directory.rglob("*")
    if file.is_file()
]

# Count the extracted files by file extension
file_extensions = (
    pd.Series(
        [
            file.suffix.lower()
            if file.suffix
            else "No extension"
            for file in all_files
        ]
    )
    .value_counts()
    .sort_index()
    .rename_axis("File Extension")
    .reset_index(name="Number of Files")
)

display(
    file_extensions.style
    .hide(axis="index")
    .set_properties(
        **{
            "text-align": "left",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                    ("vertical-align", "top"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("width", "100%"),
                ],
            },
        ]
    )
)

File Extension,Number of Files
.csv,1
.mid,3000
.mu2,3000
.pdf,2999
.txt,3000
.xml,3000


# Dataset Acquisition Summary

This notebook established a reproducible workflow for acquiring the SymbTr v3.0 dataset from the official Zenodo repository. The dataset metadata and published resources were inspected, the required files were downloaded, and the compressed archives were extracted into the project's local directory structure.

The resulting dataset is now prepared for analysis. The next notebook introduces the extracted symbolic music collections and begins a detailed exploration of the symbolic TXT representations, which serve as the primary data source for the initial statistical analyses. Later notebooks extend the analyses to additional symbolic formats, including MusicXML.

In [24]:
from IPython.display import Markdown, display

display(
    Markdown(
        """
### Notebook Completed Successfully

The SymbTr v3.0 dataset has been successfully acquired from the official Zenodo repository. The published resources were downloaded, verified, and extracted into the project's local directory structure. The extracted symbolic datasets are now available in the **interim** data directory and are ready for preprocessing and subsequent computational analyses.

"""
    )
)


### Notebook Completed Successfully

The SymbTr v3.0 dataset has been successfully acquired from the official Zenodo repository. The published resources were downloaded, verified, and extracted into the project's local directory structure. The extracted symbolic datasets are now available in the **interim** data directory and are ready for preprocessing and subsequent computational analyses.



## Next Chapter

The next chapter, **Individual Composition Analysis of the SymbTr TXT Dataset**, focuses on the analysis of individual symbolic music files. Readers will learn how to explore the structure of SymbTr TXT files and extract musical information through reproducible Python-based analyses.